# Urban Traffic Network Resilience & Bottlenecks Recipe

This recipe combines 3 distinct `algebrax` tools to evaluate city transportation grid performance:

1. **Tropical Matrix Powers** (`algebrax.semiring.TropicalSemiring` & `algebrax.matrix.core.power`):
   Computes multi-step shortest travel time latencies using Tropical matrix powers $(A \otimes B)_{i,j} = \min_k (A_{i,k} + B_{k,j})$.
2. **Forman-Ricci Curvature** (`algebrax.analysis.forman_ricci_curvature`):
   Identifies structural choke points and bridge highways ($K < 0$).
3. **Markov Steady State** (`algebrax.probability.markov_steady_state`):
   Predicts long-term vehicle density and equilibrium traffic distribution.

In [ ]:
from algebrax.analysis import forman_ricci_curvature
from algebrax.matrix.core import power
from algebrax.probability import markov_steady_state
from algebrax.semiring import TropicalSemiring

# City Transportation Graph (Travel Times in Minutes)
city_network = {
    0: {1: 8.0, 2: 12.0, 3: 15.0},
    1: {0: 8.0, 2: 6.0, 4: 20.0},
    2: {0: 12.0, 1: 6.0, 3: 5.0},
    3: {0: 15.0, 2: 5.0, 4: 10.0},
    4: {1: 20.0, 3: 10.0},
}

hub_names = {0: 'Downtown', 1: 'North Suburb', 2: 'East Industrial', 3: 'South Port', 4: 'West Airport'}

## 1. Multi-Step Shortest Path Latencies (Tropical Semiring)

Tropical matrix powers evaluate global minimal travel times.

In [ ]:
tropical_semiring = TropicalSemiring()
latency_2step = power(city_network, 2, semiring=tropical_semiring)
latency_3step = power(city_network, 3, semiring=tropical_semiring)

print('2-Step Shortest Path Travel Times:')
for u in sorted(latency_2step.keys()):
    for v, t in sorted(latency_2step[u].items()):
        if t != float('inf'):
            print(f'  Hub {u} [{hub_names[u]}] -> Hub {v} [{hub_names[v]}]: {t:.1f} min')

dt_to_airport = latency_3step.get(0, {}).get(4, float('inf'))
print(f'\nDowntown -> West Airport 3-Step Travel Time: {dt_to_airport:.1f} minutes')

## 2. Bottleneck Isolation (Forman-Ricci Curvature)

Edges with negative curvature ($K < 0$) represent critical bottleneck choke points.

In [ ]:
curvatures = forman_ricci_curvature(city_network)

print('Edge Curvature Audit Results:')
for (u, v), k in sorted(curvatures.items()):
    tag = 'CRITICAL BOTTLENECK' if k < 0 else 'Well-Connected'
    print(f'  Road ({u} <-> {v}) [{hub_names[u]} <-> {hub_names[v]}]: K = {k:+.4f} [{tag}]')

## 3. Equilibrium Traffic Distribution (Markov Steady State)

Normalizing inverse travel times into transition probabilities yields the equilibrium stationary distribution $\pi = \pi P$.

In [ ]:
markov_transition = {}
for u, neighbors in city_network.items():
    inv_tot = sum(1.0 / w for w in neighbors.values())
    markov_transition[u] = {v: (1.0 / w) / inv_tot for v, w in neighbors.items()}

steady_state = markov_steady_state(markov_transition)

print('Stationary Vehicle Distribution Across City Hubs:')
for node, prob in sorted(steady_state.items()):
    print(f'  Hub {node} [{hub_names[node]}]: {prob * 100:.2f}%')